In [5]:
import pandas as pd
import numpy as np

# RUCC 2013 codes for your specific target FIPS
# 1-3 = Metro, 4-9 = Non-metro
RUCC_MAP = {
    # Black Hills (Reference)
    '46019': 6, '46093': 3, '46081': 4, '46103': 3, '46033': 6, '46102': 9, '46047': 6,
    # Sioux Falls
    '46099': 3, '46083': 3, '46087': 6, '46125': 6, '27133': 7,
    # Billings
    '30009': 6, '30095': 6, '30111': 3,
    # Flagstaff
    '04005': 3,
    # Missoula
    '30061': 7, '30063': 3
}

# Regional grouping for filtering
BENCHMARK_GROUPS = {
    'Black Hills, SD (ref)': ['46019', '46093', '46081', '46103', '46033', '46102', '46047'],
    'Sioux Falls, SD+MN': ['46099', '46083', '46087', '46125', '27133'],
    'Billings, MT': ['30009', '30095', '30111'],
    'Flagstaff, AZ': ['04005'],
    'Missoula, MT': ['30061', '30063']
}

In [6]:
def process_irs_file(file_path):
    # Use 'latin1' to avoid the UnicodeDecodeError you encountered
    df = pd.read_csv(file_path, encoding='latin1', low_memory=False)
    
    # 1. Create 5-digit FIPS (State FIPS + County FIPS)
    # Ensuring STATEFIPS is 2 digits and COUNTYFIPS is 3 digits
    df['fips_5'] = (df['STATEFIPS'].astype(str).str.zfill(2) + 
                    df['COUNTYFIPS'].astype(str).str.zfill(3))
    
    # 2. Filter only for our Target Counties
    target_fips = list(RUCC_MAP.keys())
    df = df[df['fips_5'].isin(target_fips)].copy()
    
    # 3. Handle Suppression (N < 10 marked as 0 by IRS)
    # We turn 0 into NaN for all 'N' (count) and 'A' (amount) columns
    val_cols = [c for c in df.columns if c.startswith(('N', 'A'))]
    df[val_cols] = df[val_cols].replace(0, np.nan)
    
    return df

# Process your two specific files
df_agi = process_irs_file('data/irs_soi/22incyallagi.csv')
df_noagi = process_irs_file('data/irs_soi/22incyallnoagi.csv')

# Combine them (keeping agi_stub to distinguish)
df_combined = pd.concat([df_agi, df_noagi], ignore_index=True)

/tmp/ipykernel_97158/463170987.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['fips_5'] = (df['STATEFIPS'].astype(str).str.zfill(2) +
/tmp/ipykernel_97158/463170987.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['fips_5'] = (df['STATEFIPS'].astype(str).str.zfill(2) +


In [7]:
# 1. Map RUCC Codes
df_combined['rucc_2013'] = df_combined['fips_5'].map(RUCC_MAP)

# 2. Map Benchmark Region Names
def get_region(fips):
    for region, fips_list in BENCHMARK_GROUPS.items():
        if fips in fips_list:
            return region
    return "Other"

df_combined['benchmark_region'] = df_combined['fips_5'].apply(get_region)

# 3. Specific Flag for Oglala Lakota (46102) charitable suppression
df_combined['charitable_suppressed_flag'] = df_combined['fips_5'] == '46102'

# Final check of the data
print(f"Total Rows: {len(df_combined)}")
print(df_combined[['fips_5', 'COUNTYNAME', 'rucc_2013', 'benchmark_region']].head())

Total Rows: 162
  fips_5       COUNTYNAME  rucc_2013 benchmark_region
0  04005  Coconino County          3    Flagstaff, AZ
1  04005  Coconino County          3    Flagstaff, AZ
2  04005  Coconino County          3    Flagstaff, AZ
3  04005  Coconino County          3    Flagstaff, AZ
4  04005  Coconino County          3    Flagstaff, AZ


/tmp/ipykernel_97158/2523589337.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_combined['rucc_2013'] = df_combined['fips_5'].map(RUCC_MAP)
/tmp/ipykernel_97158/2523589337.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_combined['benchmark_region'] = df_combined['fips_5'].apply(get_region)
/tmp/ipykernel_97158/2523589337.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

In [8]:
# Type Hardening: Convert objects to string for Parquet
for col in df_combined.select_dtypes('object').columns:
    df_combined[col] = df_combined[col].astype(str)

df_combined.to_parquet('processed_data/irs_soi_benchmark_combined.parquet', index=False)
df_combined.to_csv('processed_data/irs_soi_benchmark_combined.csv', index=False)

/tmp/ipykernel_97158/2919866170.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_combined.select_dtypes('object').columns:
